# Data Gathering Demos

This notebook contains standalone data-gathering examples for the Real World Data Wrangling presentation. These demos create or load data that is not used by the main cleaning and analysis notebook, so they live here instead of interrupting the project-flow notebook.

## Setup

Keep this notebook in the same folder as `library_program_registrations_dirty.csv` and `library_branch_lookup.csv` so the file-path examples work.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile

import pandas as pd

DATA_DIR = Path(".")
programs_path = DATA_DIR / "library_program_registrations_dirty.csv"
branches_path = DATA_DIR / "library_branch_lookup.csv"

print("Program dataset exists:", programs_path.exists())
print("Branch dataset exists:", branches_path.exists())

programs_raw = pd.read_csv(programs_path)
branches_raw = pd.read_csv(branches_path)
print("Program rows:", len(programs_raw))
print("Branch rows:", len(branches_raw))


### Files on Hand: CSV, ZIP, JSON, and Excel Notes

The source files are CSVs, but this cell also demonstrates how ZIP and JSON can fit into the same workflow. Excel files follow the same idea with `pd.read_excel("file.xlsx")`, although this demo avoids requiring the extra `openpyxl` package.

In [8]:
# Demonstrate a ZIP file workflow using one of the existing CSV datasets.
archive_path = DATA_DIR / "demo_files_on_hand.zip"
with zipfile.ZipFile(archive_path, mode="w") as archive:
    archive.write(programs_path, arcname=programs_path.name)

with zipfile.ZipFile(archive_path) as archive:
    print("Files inside ZIP:", archive.namelist())

programs_from_zip = pd.read_csv(archive_path)
programs_from_zip.head(3)

Files inside ZIP: ['library_program_registrations_dirty.csv']


,registration_id,learner_name,Age,city,branch_id,program_id,program_name,session_date,program_type,interests,registration_status,attendance_hours,pre_score,post_score,fee_paid,scholarship_amount,empty_column
0,R001,Emma Johnson,8,New York,B101,P001,Intro to Python,2026-01-12,STEM,coding; robots,Attended,2.0,45,72,$0,$25,NaN
1,R002,Liam Smith,10,Los Angeles,B102,P001,Intro to Python,01/20/2026,STEM,coding; games,Attended,2.5,50,80,$0,$25,NaN
2,R003,Olivia Brown,11,Chicago,B103,P002,Data Storytelling,Feb 02 2026,Data,charts; storytelling,No Show,0,60,*,$5,$0,NaN


In [9]:
# Demonstrate JSON by saving and reloading the branch lookup as records.
branches_json_path = DATA_DIR / "branch_lookup_demo.json"
branches_raw.to_json(branches_json_path, orient="records", indent=2)
branches_from_json = pd.read_json(branches_json_path)
branches_from_json.head(3)

,branch_id,branch_name,branch_city,state,region,opened_year,square_feet,annual_visits,wifi_available
0,B101,Central Library,New York,NY,Northeast,1911,80000,"1,250,000",True
1,B102,Westside Branch,Los Angeles,CA,West,1954,45000,"850,000",True
2,B103,Lakeview Branch,Chicago,IL,Midwest,1972,38000,"620,000",True


### Optional API Example

This uses a no-key US ZIP-code API endpoint to show the `requests.get()` pattern. If the network is unavailable, the notebook prints the reason and continues.

In [10]:
import requests

api_url = "https://api.zippopotam.us/us/90210"
try:
    response = requests.get(api_url, timeout=10, headers={"User-Agent": "udacity-demo"})
    response.raise_for_status()
    payload = response.json()
    zip_lookup = pd.DataFrame(payload["places"])
    zip_lookup["post_code"] = payload["post code"]
    zip_lookup[["post_code", "place name", "state", "latitude", "longitude"]]
except Exception as exc:
    zip_lookup = pd.DataFrame()
    print("API demo skipped:", exc)

### Optional Kaggle API Example

For the Kaggle slide, use a non-movie topic so learners can still choose the movie dataset for their own project.

Suggested dataset: `camnugent/california-housing-prices`

Topic: California housing prices from census-based district data.

This section only downloads when the Kaggle CLI and credentials are configured. Otherwise it prints the exact command to use.

In [11]:
# Optional install command if the Kaggle CLI is missing:
# python -m pip install kaggle==1.6.12

kaggle_dataset = "camnugent/california-housing-prices"
kaggle_dir = DATA_DIR / "kaggle_california_housing"
kaggle_file = kaggle_dir / "housing.csv"
kaggle_cmd = [
    "kaggle", "datasets", "download",
    "-d", kaggle_dataset,
    "-p", str(kaggle_dir),
    "--unzip",
]

kaggle_config = Path.home() / ".kaggle" / "kaggle.json"
print("Kaggle dataset:", kaggle_dataset)
print("Download command:", " ".join(kaggle_cmd))

if kaggle_file.exists():
    housing = pd.read_csv(kaggle_file)
    print("Loaded existing Kaggle file:", kaggle_file)
    print(housing.shape)
elif shutil.which("kaggle") and kaggle_config.exists():
    kaggle_dir.mkdir(exist_ok=True)
    result = subprocess.run(kaggle_cmd, check=False, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0 and kaggle_file.exists():
        housing = pd.read_csv(kaggle_file)
        print("Downloaded and loaded:", housing.shape)
    else:
        housing = pd.DataFrame()
        print("Kaggle download did not complete. Error output:")
        print(result.stderr)
else:
    housing = pd.DataFrame()
    print("Kaggle CLI or credentials are not configured here. Use the printed command in a configured workspace.")

Kaggle dataset: camnugent/california-housing-prices
Download command: kaggle datasets download -d camnugent/california-housing-prices -p kaggle_california_housing --unzip
Kaggle CLI or credentials are not configured here. Use the printed command in a configured workspace.
